In [1]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from sklearn.metrics import f1_score

In [2]:
vectorizer = TfidfVectorizer(
    token_pattern=r"\d+",
    ngram_range=(1, 3),
    max_features=50000
)

relieable = ["reliable", "political", "clickbait", "unreliable"]
# unrelieable_score = {
#     'fake' : 1.0,
#     'conspiracy' : 1.0,
#     'rumor' : 0.9,
#     'hate' : 0.9,
#     'satire' : 0.9,
#     'junksci' : 0.85,
#     'bias' : 0.8,
#     'unreliable' : 0.35,
#     'clickbait' : 0.15,
#     'political' : 0.05,
#     'reliable' : 0.0,
# }
def type_to_value(t : str):
    return 0 if t in relieable else 1

def extract_df(df : pd.DataFrame, is_train : bool = False):
    print("Categorizing type...")
    y = df['type'].map(type_to_value)

    print("Vectorizing content tokens...")
    if is_train: X_content = vectorizer.fit_transform(df["tokens"])
    else: X_content = vectorizer.transform(df["tokens"])

    return X_content, y

In [3]:
train_df = pd.read_csv("data/final_dataset/splits/train.csv")
X_train, y_train = extract_df(train_df, is_train=True)

X_val, y_val = extract_df(pd.read_csv("data/final_dataset/splits/val.csv"))

Categorizing type...
Vectorizing content tokens...
Categorizing type...
Vectorizing content tokens...


In [4]:
y_train = train_df['type'].map(type_to_value)

In [5]:
from sklearn.linear_model import LogisticRegression

logre_model = LogisticRegression(max_iter=500, C=5, class_weight='balanced')
logre_model.fit(X_train, y_train)

print(f1_score(y_val, logre_model.predict(X_val)))

# 805 -> 823 -> 838 -> 847 ->rel 872

0.8720271207738322


In [6]:
mnb_model = MultinomialNB(alpha=1e-100)

mnb_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, mnb_model.predict(X_val))}")

f1 score: 0.8249508067689886


In [7]:
cnb_model = ComplementNB(alpha=1e-100)

cnb_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, cnb_model.predict(X_val))}")

f1 score: 0.8269176081280215


c:\Users\andre\KU\GDS\GDS2026\.venv\Lib\site-packages\sklearn\naive_bayes.py:1077: RuntimeWarning: divide by zero encountered in log
  logged = np.log(comp_count / comp_count.sum(axis=1, keepdims=True))


In [8]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(penalty='l1', C=1, class_weight='balanced')
svm_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, svm_model.predict(X_val))}")

# 808 -> 829 -> 841 -> 847 | 852 ->rel 876

f1 score: 0.8756456306861383


In [14]:
import joblib

joblib.dump(svm_model, "data/models/linearSVC.joblib")
joblib.dump(vectorizer, "data/models/linearSVC_vectorizer.joblib")

['data/models/linearSVC_vectorizer.joblib']

In [10]:
# import json

# with open("data/models/linearSVC.json", 'w') as f:
#     json.dump({
#     'classes' : svm_model.classes_.tolist(),
#     'intercept' : svm_model.intercept_.tolist(),
#     'coef' : svm_model.coef_.tolist(),
# }, f, indent=4)

# vectorizer.get

In [11]:
# model = LinearSVC()
# model.coef_ = svm_model.coef_
# model.intercept_ = svm_model.intercept_
# model.classes_ = svm_model.classes_

# print(f"f1 score: {f1_score(y_val, model.predict(X_val))}")